# 05 - LangChain 文档加载器（Document Loaders）

## 学习目标
- 掌握 PyPDFLoader 加载 PDF 文档并提取元数据
- 使用 DirectoryLoader 递归加载多种文件格式
- 使用 WebBaseLoader 从网页抓取内容并控制速率
- 使用 RecursiveUrlLoader 爬取网站
- 使用 UnstructuredMarkdownLoader 保留 Markdown 结构
- 合并多个加载器并去重（MergedDataLoader）
- 理解惰性加载模式 lazy_load() vs load()

In [ ]:
# 安装必要依赖（如未安装请取消注释）
# !pip install langchain langchain-community pypdf unstructured markdown pdfplumber langchain-text-splitters

## 1. PyPDFLoader - 加载 PDF 文档

PyPDFLoader 使用 pypdf 库加载 PDF，支持按页分割和元数据提取。

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# 创建示例 PDF 文件用于演示
import os
sample_pdf_path = "sample_document.pdf"

# 如果没有真实的 PDF 文件，我们创建一个简单的示例说明
print("=== PyPDFLoader 使用示例 ===\n")

# 加载 PDF
# loader = PyPDFLoader("/path/to/your/document.pdf")
# documents = loader.load()

# 示例代码（假设有 PDF 文件）:
print("# 基本用法:")
print("loader = PyPDFLoader('document.pdf')")
print("docs = loader.load()")
print()

# PyPDFLoader 支持的参数
print("# 高级参数:")
print("loader = PyPDFLoader(")
print("    file_path='document.pdf',")
print("    extract_images=False,          # 是否提取图片")
print("    headers=None,                   # 自定义 HTTP 请求头")
print("    extraction_mode='plain',        # 'plain' 或 'layout'")
print("    password=None                   # 加密 PDF 的密码")
print(")")
print()

# 元数据提取示例
print("# 每页文档包含的元数据:")
print("# - source: 文件路径")
print("# - page: 页码（从0开始）")
print("# - total_pages: 总页数")
print("# - PDF 信息: Author, Creator, Producer, Subject, Title")
print()

print("# 查看文档数量和第一块内容预览:")
print("print(f'总页数: {len(docs)}')")
print("print(f'第1页预览: {docs[0].page_content[:200]}...')")
print("print(f'第1页元数据: {docs[0].metadata}')")


In [ ]:
# 实际运行示例: 创建一个临时 PDF 并加载
try:
    # 使用 PyPDFLoader 加载一个小的测试 PDF
    # 这里我们尝试加载实际文件，如果不存在则创建
    
    # 为了演示，我们创建一个最简单的 PDF（需要 reportlab）
    try:
        from reportlab.pdfgen import canvas
        from reportlab.lib.pagesizes import A4
        
        test_pdf = "test_sample.pdf"
        c = canvas.Canvas(test_pdf, pagesize=A4)
        c.drawString(100, 750, "第一章：LangChain 文档加载器入门")
        c.drawString(100, 730, "LangChain 提供了丰富的文档加载器，支持从多种来源加载文档。")
        c.drawString(100, 710, "支持的格式包括 PDF、Word、Markdown、网页、CSV 等。")
        c.showPage()
        c.drawString(100, 750, "第二章：向量存储与检索")
        c.drawString(100, 730, "向量存储是 RAG 系统的核心组件，负责存储和检索文档嵌入。")
        c.drawString(100, 710, "常用的向量数据库有 Chroma、Qdrant、Pinecone、Weaviate 等。")
        c.showPage()
        c.drawString(100, 750, "第三章：检索器与路由")
        c.drawString(100, 730, "检索器负责根据查询从向量存储中检索相关文档。")
        c.drawString(100, 710, "高级检索策略包括 MMR、SelfQuery、多查询检索等。")
        c.save()
        print(f"已创建测试 PDF: {test_pdf}")
        
        # 加载 PDF
        loader = PyPDFLoader(test_pdf)
        documents = loader.load()
        
        print(f"\n总页数: {len(documents)}")
        for i, doc in enumerate(documents):
            print(f"\n--- 第 {i+1} 页 ---")
            print(f"内容预览: {doc.page_content[:100]}")
            print(f"元数据: {doc.metadata}")
        
        # 清理测试文件
        os.remove(test_pdf)
        
    except ImportError:
        print("reportlab 未安装，跳过 PDF 创建演示")
        print("请安装: pip install reportlab")
        print("\n以下为概念性代码演示:")
        print("=" * 50)
        print("# 模拟加载结果")
        print("loader = PyPDFLoader('document.pdf')")
        print("docs = loader.load()")
        print("# 返回: [Document(page_content='...', metadata={'source': 'document.pdf', 'page': 0}), ...]")
        
except Exception as e:
    print(f"运行时错误: {e}")
    print("这是预期行为 - 需要实际的 PDF 文件才能运行")

## 2. DirectoryLoader - 递归加载目录中的文档

DirectoryLoader 支持使用 glob 模式递归加载指定目录中的文件，支持多种格式。

In [ ]:
import tempfile
import os

# 创建测试目录结构
test_dir = tempfile.mkdtemp(prefix="langchain_docs_")
os.makedirs(os.path.join(test_dir, "subdir"), exist_ok=True)

# 创建测试文件
files_content = {
    "intro.txt": "LangChain is a framework for developing applications powered by language models.\nIt provides document loaders, vector stores, and retrievers.",
    "guide.md": "# LangChain Guide\n\n## Document Loaders\n\nDocument loaders load data from many sources.\n\n## Vector Stores\n\nVector stores store embeddings.",
    "subdir/notes.txt": "Advanced techniques include multi-query retrieval and self-query retrieval.\nThese improve retrieval quality significantly.",
    "config.yaml": "model: gpt-4\ntemperature: 0.7\nmax_tokens: 2000",
}

for filename, content in files_content.items():
    filepath = os.path.join(test_dir, filename)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

print(f"测试目录: {test_dir}")
print("\n文件结构:")
for root, dirs, files in os.walk(test_dir):
    level = root.replace(test_dir, "").count(os.sep)
    indent = "  " * level
    folder = os.path.basename(root) if root != test_dir else "."
    print(f"{indent}{folder}/")
    for file in files:
        print(f"{indent}  {file}")

In [ ]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import UnstructuredMarkdownLoader

# === 示例1: 加载所有 .txt 文件（非递归） ===
print("=" * 60)
print("示例1: 加载所有 .txt 文件（非递归）")
print("=" * 60)

loader = DirectoryLoader(
    path=test_dir,
    glob="**/*.txt",           # glob 模式: 递归匹配所有 .txt
    loader_cls=TextLoader,      # 使用 TextLoader 加载每个文件
    loader_kwargs={"autodetect_encoding": True},
    show_progress=True,         # 显示进度条
    use_multithreading=True,    # 多线程加载
    max_concurrency=4,          # 最大并发数
    silent_errors=False,        # 是否静默错误
    sample_size=0,              # 采样大小（0=全部）
    randomize_sample=False,     # 是否随机采样
)

txt_docs = loader.load()
print(f"\n加载了 {len(txt_docs)} 个 .txt 文档:")
for doc in txt_docs:
    source = doc.metadata.get("source", "unknown")
    preview = doc.page_content[:80].replace("\n", " ")
    print(f"  [{os.path.basename(source)}] {preview}...")


In [ ]:
# === 示例2: 加载多种文件类型 ===
print("\n" + "=" * 60)
print("示例2: 加载 .md 文件（保留结构）")
print("=" * 60)

md_loader = DirectoryLoader(
    path=test_dir,
    glob="**/*.md",
    loader_cls=UnstructuredMarkdownLoader,
    loader_kwargs={"mode": "elements"},  # 按元素（标题、段落等）分割
    show_progress=True,
)

md_docs = md_loader.load()
print(f"\n加载了 {len(md_docs)} 个 .md 文档:")
for doc in md_docs:
    source = doc.metadata.get("source", "unknown")
    preview = doc.page_content[:100].replace("\n", " ")
    category = doc.metadata.get("category", "")
    print(f"  [{os.path.basename(source)}] ({category}) {preview}...")


In [ ]:
# === 示例3: 文件过滤 - 只加载特定文件 ===
print("\n" + "=" * 60)
print("示例3: 文件过滤")
print("=" * 60)

# 使用 loader_kwargs 传递过滤函数
from pathlib import Path

def file_filter(file_path: str) -> bool:
    """只加载内容长度大于 50 字符的文件"""
    return os.path.getsize(file_path) > 50

loader_filtered = DirectoryLoader(
    path=test_dir,
    glob="**/*",
    loader_cls=TextLoader,
    loader_kwargs={"autodetect_encoding": True},
    silent_errors=True,  # 跳过无法加载的文件（如 .yaml 用 TextLoader 可能出错）
)

# 手动过滤
all_docs = []
for file_path in Path(test_dir).glob("**/*"):
    if file_path.is_file() and file_filter(str(file_path)):
        try:
            loader_single = TextLoader(str(file_path), autodetect_encoding=True)
            all_docs.extend(loader_single.load())
        except Exception:
            pass  # 跳过无法加载的文件

print(f"过滤后加载了 {len(all_docs)} 个文档（原目录有 {len(list(Path(test_dir).glob('**/*'))) - 3} 个文件，减去 3 个子目录）")
for doc in all_docs:
    source = doc.metadata.get("source", "unknown")
    print(f"  {os.path.basename(source)}")


## 3. WebBaseLoader - 从网页抓取内容

WebBaseLoader 用于从 URL 抓取网页内容，支持自定义请求头和速率限制。

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

print("=== WebBaseLoader 使用示例 ===\n")

# === 示例1: 基本用法 - 加载单个 URL ===
print("示例1: 加载单个 URL")
print("-" * 40)

# 使用 requests_per_second 控制速率
loader = WebBaseLoader(
    web_paths=["https://python.langchain.com/docs/introduction/"],
    header_template={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    },
    requests_per_second=2,    # 每秒最多2个请求
    requests_kwargs={
        "timeout": 10,         # 请求超时
        "verify": True,       # SSL 验证
    },
    default_parser="html.parser",  # BeautifulSoup 解析器
    continue_on_failure=True,       # 遇到失败继续
    show_progress=True,
)

try:
    web_docs = loader.load()
    print(f"\n加载了 {len(web_docs)} 个网页文档")
    if web_docs:
        first_doc = web_docs[0]
        print(f"来源: {first_doc.metadata.get('source', 'N/A')}")
        print(f"标题: {first_doc.metadata.get('title', 'N/A')}")
        print(f"内容预览 ({len(first_doc.page_content)} 字符):")
        print(first_doc.page_content[:300] + "...")
except Exception as e:
    print(f"网络请求失败（预期 - 可能需要网络访问）: {type(e).__name__}")
    print(f"错误详情: {e}")

print("\n" + "=" * 60)

# === 示例2: 加载多个 URL ===
print("示例2: 加载多个 URL（带速率限制）")
print("-" * 40)

loader_multi = WebBaseLoader(
    web_paths=[
        "https://python.langchain.com/docs/introduction/",
        "https://python.langchain.com/docs/concepts/",
    ],
    requests_per_second=1.0,  # 每秒1个请求
    continue_on_failure=True,
)

print("loader_multi = WebBaseLoader(web_paths=[...], requests_per_second=1.0)")
print("# requests_per_second 确保不会过快发送请求，避免被封")


## 4. RecursiveUrlLoader - 递归爬取网站

RecursiveUrlLoader 从指定 URL 开始，递归爬取子链接，直到达到最大深度。

In [ ]:
from langchain_community.document_loaders import RecursiveUrlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("=== RecursiveUrlLoader 使用示例 ===\n")

# RecursiveUrlLoader 参数说明
print("# RecursiveUrlLoader 参数:")
print("loader = RecursiveUrlLoader(")
print("    url='https://example.com/docs',     # 起始 URL")
print("    max_depth=2,                        # 最大爬取深度")
print("    use_async=False,                    # 是否使用异步")
print("    extractor=None,                     # 自定义内容提取器")
print("    metadata_extractor=None,            # 自定义元数据提取器")
print("    exclude_dirs=[],                    # 排除的目录")
print("    timeout=10,                         # 请求超时")
print("    prevent_outside=True,               # 防止爬出域名")
print("    link_regex='',                      # 链接提取正则")
print("    headers={},                         # 自定义请求头")
print("    check_response_status=True,         # 检查响应状态")
print("    continue_on_failure=True,           # 遇到失败继续")
print(")")

print("\n# 自定义内容提取器示例:")
print("from bs4 import BeautifulSoup")
print("def simple_extractor(html_content: str) -> str:")
print("    soup = BeautifulSoup(html_content, 'html.parser')")
print("    # 移除脚本和样式")
print("    for tag in soup(['script', 'style', 'nav', 'footer']):")
print("        tag.decompose()")
print("    return soup.get_text()")

# 实际演示（使用本地文件模拟）
print("\n# 由于网络限制，以下是概念性代码演示:")
print("loader = RecursiveUrlLoader(")
print("    url='https://python.langchain.com/docs/',")
print("    max_depth=2,")
print("    prevent_outside=True,")
print(")")
print("docs = loader.load()")
print("print(f'爬取了 {len(docs)} 个页面')")
print("for doc in docs[:3]:")
print("    print(f'  URL: {doc.metadata[\"source\"]}')")
print("    print(f'  内容长度: {len(doc.page_content)}')")


## 5. UnstructuredMarkdownLoader - 加载 Markdown 保留结构

UnstructuredMarkdownLoader 可以识别 Markdown 的标题、代码块、列表等结构元素。

In [ ]:
from langchain_community.document_loaders import UnstructuredMarkdownLoader

# 创建示例 Markdown 文件
sample_md_path = os.path.join(test_dir, "advanced_guide.md")
sample_md_content = """# LangChain 高级指南

## 检索增强生成 (RAG)

RAG 结合了检索和生成两个步骤。

### 检索步骤

1. 将文档分块并嵌入
2. 存储到向量数据库
3. 查询时检索最相似的块

### 生成步骤

将检索到的上下文注入提示词，由 LLM 生成回答。

```python
from langchain.chains import RetrievalQA
chain = RetrievalQA.from_chain_type(llm, retriever=retriever)
```

## 高级检索策略

| 策略 | 描述 | 使用场景 |
|------|------|----------|
| MMR | 最大边际相关性 | 需要多样性 |
| SelfQuery | 自查询 | 元数据过滤 |
| MultiQuery | 多查询 | 查询改写 |

> **注意**: 选择合适的检索策略可以显著提升 RAG 系统质量。
"""

with open(sample_md_path, "w", encoding="utf-8") as f:
    f.write(sample_md_content)

print("=== UnstructuredMarkdownLoader 使用示例 ===\n")

# === 模式1: single - 整个文件作为一个文档 ===
print("模式1: mode='single' - 整个文件作为一个文档")
print("-" * 50)
loader_single = UnstructuredMarkdownLoader(sample_md_path, mode="single")
single_docs = loader_single.load()
print(f"文档数: {len(single_docs)}")
print(f"内容长度: {len(single_docs[0].page_content)} 字符")
print(f"元数据: {single_docs[0].metadata}")

print("\n" + "=" * 60)

# === 模式2: elements - 按结构元素分割 ===
print("模式2: mode='elements' - 按结构元素分割（标题、段落、代码块等）")
print("-" * 50)
loader_elements = UnstructuredMarkdownLoader(sample_md_path, mode="elements")
element_docs = loader_elements.load()
print(f"文档数: {len(element_docs)}")
for i, doc in enumerate(element_docs):
    category = doc.metadata.get("category", "unknown")
    preview = doc.page_content[:60].replace("\n", " ")
    print(f"  [{i}] ({category}) {preview}...")

print("\n" + "=" * 60)

# === 模式3: paged - 按页分割 ===
print("模式3: mode='paged' - 按页分割")
print("-" * 50)
loader_paged = UnstructuredMarkdownLoader(sample_md_path, mode="paged")
paged_docs = loader_paged.load()
print(f"文档数: {len(paged_docs)}")
for i, doc in enumerate(paged_docs):
    print(f"  页面 {doc.metadata.get('page_number', '?')}: {len(doc.page_content)} 字符")

# 清理
os.remove(sample_md_path)

## 6. MergedDataLoader - 合并多个加载器

MergedDataLoader 将多个加载器的结果合并，并自动去重。

In [ ]:
from langchain_community.document_loaders.merge import MergedDataLoader
from langchain_community.document_loaders import TextLoader

print("=== MergedDataLoader 使用示例 ===\n")

# 创建一些测试文件
files_for_merge = [
    ("data1.txt", "这是来自文件1的内容。包含一些重复的信息。"),
    ("data2.txt", "这是来自文件2的内容。包含一些重复的信息。"),
    ("data3.txt", "这是来自文件3的独特内容。"),
]

merge_paths = []
for fname, fcontent in files_for_merge:
    fpath = os.path.join(test_dir, fname)
    with open(fpath, "w", encoding="utf-8") as f:
        f.write(fcontent)
    merge_paths.append(fpath)

# 创建多个加载器
loaders = [TextLoader(fp, autodetect_encoding=True) for fp in merge_paths]

print(f"创建了 {len(loaders)} 个加载器")

# 使用 MergedDataLoader 合并
merged_loader = MergedDataLoader(loaders=loaders)
merged_docs = merged_loader.load()

print(f"\n合并后文档总数: {len(merged_docs)}")
for doc in merged_docs:
    source = doc.metadata.get("source", "unknown")
    print(f"  [{os.path.basename(source)}] {doc.page_content[:60]}...")

# 清理
for fp in merge_paths:
    os.remove(fp)

## 7. 惰性加载模式: lazy_load() vs load()

惰性加载可以处理大规模文档，避免一次性将所有文档加载到内存中。

In [ ]:
from langchain_community.document_loaders import TextLoader

print("=== lazy_load() vs load() 对比 ===\n")

# 创建一个较大的测试文件
large_file_path = os.path.join(test_dir, "large_document.txt")
with open(large_file_path, "w", encoding="utf-8") as f:
    for i in range(100):
        f.write(f"第 {i+1} 行: 这是 LangChain 文档加载器测试内容。文档加载器负责从各种来源加载文档。\n")

print(f"测试文件: {large_file_path}")
print(f"文件大小: {os.path.getsize(large_file_path)} 字节\n")

# === 方法1: load() - 一次性加载全部 ===
print("方法1: loader.load() - 一次性加载全部到内存")
print("-" * 50)
loader_all = TextLoader(large_file_path, autodetect_encoding=True)
all_docs = loader_all.load()
print(f"一次性加载: {len(all_docs)} 个文档")
print(f"第一个文档长度: {len(all_docs[0].page_content)} 字符")
print(f"内存使用: 所有文档同时在内存中")

print("\n" + "=" * 60)

# === 方法2: lazy_load() - 惰性加载（逐文档产出）===
print("方法2: loader.lazy_load() - 惰性加载，逐个产出")
print("-" * 50)

loader_lazy = TextLoader(large_file_path, autodetect_encoding=True)

print("# 惰性加载不会立即将所有文档读入内存")
print("# 适用于超大文件（GB级别）或大量文件")
print()

doc_count = 0
total_chars = 0
for doc in loader_lazy.lazy_load():
    doc_count += 1
    total_chars += len(doc.page_content)
    if doc_count <= 3:
        preview = doc.page_content[:50].strip()
        print(f"  [文档 {doc_count}] {preview}...")

print(f"\n惰性加载完成: {doc_count} 个文档, 共 {total_chars} 字符")
print(f"优势: 每个文档只在迭代到它时才加载，内存使用恒定")

print("\n" + "=" * 60)

# === 实际场景对比 ===
print("实际场景对比:")
print("-" * 50)
print()
print("场景: 加载 1000 个 PDF 文件，共 10GB")
print()
print("load() 方法:")
print("  - 一次性读入所有 1000 个文件")
print("  - 内存峰值: ~10GB+")
print("  - 适用: 小规模数据集，需要随机访问所有文档")
print()
print("lazy_load() 方法:")
print("  - 逐个文件加载，处理完即可释放")
print("  - 内存峰值: ~10MB（单个文件）")
print("  - 适用: 大规模数据集，流式处理，嵌入批处理")

# 清理
os.remove(large_file_path)

In [ ]:
# 清理测试目录
import shutil
shutil.rmtree(test_dir, ignore_errors=True)
print(f"\n已清理测试目录: {test_dir}")

## 总结

| 加载器 | 用途 | 关键参数 |
|--------|------|----------|
| PyPDFLoader | 加载 PDF | extraction_mode, password |
| DirectoryLoader | 批量加载文件 | glob, loader_cls, show_progress |
| WebBaseLoader | 抓取网页 | requests_per_second, header_template |
| RecursiveUrlLoader | 爬取网站 | max_depth, prevent_outside |
| UnstructuredMarkdownLoader | 加载 MD | mode (single/elements/paged) |
| MergedDataLoader | 合并加载器 | loaders 列表 |
| lazy_load() | 惰性加载 | 迭代器模式，节省内存 |